# Praktikum 4 - Perbandingan Annoy, FAISS, dan HNSW

**Nama:** Agus

**NIM:** 244107020049

[Sumber modul](https://polinema.gitbook.io/jti-modul-praktikum-pembelajaran-mesin-2025-2026/js06-ann-approximate-nearest-neighbors/praktikum-4)

## Langkah 0 - Persiapan Lingkungan

In [1]:
import sys
import numpy as np
np.random.seed(42)
print("Kernel:", sys.executable)

import faiss
faiss.omp_set_num_threads(4)


Kernel: /Volumes/External/miniconda3/envs/ml-praktikum/bin/python


## Langkah 1 - Import Library

In [2]:
import numpy as np
import time
from annoy import AnnoyIndex
import faiss
import hnswlib

# ===============================


## Langkah 2 - Buat dataset 1 juta data 5D

In [3]:
# 1. Buat dataset 1 juta data 5D
# ===============================
n_data = 1_000_000   # bisa coba 100_000 dulu jika RAM terbatas
dim = 5
X = np.random.random((n_data, dim)).astype(np.float32)

# Query point
query = np.random.random((1, dim)).astype(np.float32)
k = 10

# ===============================


## Langkah 3 - Annoy

In [4]:
# 2. Annoy
# ===============================
print("=== Annoy ===")
ann_index = AnnoyIndex(dim, 'euclidean')

start = time.time()
for i in range(n_data):
    ann_index.add_item(i, X[i])
ann_index.build(10)  # 10 trees
build_time = time.time() - start

start = time.time()
neighbors = ann_index.get_nns_by_vector(query[0], k, include_distances=True)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", neighbors[0][:5], "...")

# ===============================


=== Annoy ===


Build time: 3.0225820541381836 detik
Query time: 0.00013589859008789062 detik
Neighbors: [901095, 495168, 561017, 163986, 185734] ...


## Langkah 4 - FAISS (Flat Index)

In [5]:
# 3. FAISS (Flat Index)
# ===============================
print("\n=== FAISS (IndexFlatL2) ===")
faiss_index = faiss.IndexFlatL2(dim)

start = time.time()
faiss_index.add(X)
build_time = time.time() - start

start = time.time()
distances, indices = faiss_index.search(query, k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", indices[0][:5], "...")

# ===============================



=== FAISS (IndexFlatL2) ===
Build time: 0.0024361610412597656 detik
Query time: 0.003297090530395508 detik
Neighbors: [901095 495168 561017 163986 185734] ...


## Langkah 5 - HNSW (hnswlib)

In [6]:
# 4. HNSW (hnswlib)
# ===============================
print("\n=== HNSW (hnswlib) ===")
hnsw_index = hnswlib.Index(space='l2', dim=dim)

start = time.time()
hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
hnsw_index.add_items(X)
build_time = time.time() - start

hnsw_index.set_ef(50)

start = time.time()
labels, distances = hnsw_index.knn_query(query, k=k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", labels[0][:5], "...")



=== HNSW (hnswlib) ===


Build time: 10.319157838821411 detik
Query time: 6.604194641113281e-05 detik
Neighbors: [901095 495168 561017 163986 185734] ...


## Percobaan Tambahan - Cosine pada Data yang Sama

Annoy Angular, FAISS inner product pada vektor L2, dan HNSW Cosine memiliki urutan kedekatan cosine yang sebanding. Nilai jarak native berbeda, sehingga recall membandingkan ID tetangga.

In [7]:
import pandas as pd
from IPython.display import display

X_l2 = X.copy()
query_l2 = query.copy()
faiss.normalize_L2(X_l2)
faiss.normalize_L2(query_l2)
cosine_rows = []
start = time.perf_counter()
flat_cosine = faiss.IndexFlatIP(dim)
flat_cosine.add(X_l2)
flat_build = time.perf_counter() - start
start = time.perf_counter()
_, gt_cosine = flat_cosine.search(query_l2, k)
flat_query = time.perf_counter() - start
cosine_rows.append({'Model': 'FAISS Flat IP (exact)', 'Build (s)': flat_build,
                   'Query (ms)': flat_query * 1000, 'Recall@10': 1.0,
                   'Index': gt_cosine[0].tolist()})

start = time.perf_counter()
angular = AnnoyIndex(dim, 'angular')
for i, vec in enumerate(X_l2):
    angular.add_item(i, vec)
angular.build(10)
angular_build = time.perf_counter() - start
start = time.perf_counter()
angular_ids = angular.get_nns_by_vector(query_l2[0], k)
angular_query = time.perf_counter() - start
cosine_rows.append({'Model': 'Annoy Angular', 'Build (s)': angular_build,
                   'Query (ms)': angular_query * 1000,
                   'Recall@10': len(set(angular_ids) & set(gt_cosine[0])) / k,
                   'Index': angular_ids})

start = time.perf_counter()
cosine_hnsw = hnswlib.Index(space='cosine', dim=dim)
cosine_hnsw.init_index(max_elements=n_data, ef_construction=200, M=16)
cosine_hnsw.add_items(X)
cosine_hnsw.set_ef(50)
cosine_build = time.perf_counter() - start
start = time.perf_counter()
cosine_ids, _ = cosine_hnsw.knn_query(query, k=k)
cosine_query = time.perf_counter() - start
cosine_rows.append({'Model': 'HNSW Cosine', 'Build (s)': cosine_build,
                   'Query (ms)': cosine_query * 1000,
                   'Recall@10': len(set(cosine_ids[0]) & set(gt_cosine[0])) / k,
                   'Index': cosine_ids[0].tolist()})
display(pd.DataFrame(cosine_rows).round(6))


,Model,Build (s),Query (ms),Recall@10,Index
0,FAISS Flat IP (exact),0.002963,1.789250,1.0,"[653774, 135588, 561017, 901095, 211473, 16398..."
1,Annoy Angular,3.205087,0.120833,1.0,"[653774, 135588, 561017, 901095, 211473, 16398..."
2,HNSW Cosine,9.614000,0.067750,1.0,"[653774, 135588, 561017, 901095, 211473, 16398..."


## Analisis

IndexFlatL2 dan IndexFlatIP adalah pencarian exact. Annoy dan HNSW adalah approximate. Build cost, query time, dan recall perlu dibaca bersama. Hasil Euclidean dan Cosine dapat memilih tetangga berbeda karena panjang vektor memengaruhi Euclidean.